# Chapter 2: Satellite Data Acquisition

In [ ]:
import pygeovision as pgv

client = pgv.PyGeoVision()

## Search providers

In [ ]:
results = client.search(
    bbox            = (-0.30, 5.50, -0.05, 5.70),
    date_range      = ("2026-06-01", "2026-06-30"),
    cloud_cover_max = 20,
)
print(f"Total: {len(results)}")
providers = {}
for r in results:
    providers[r.provider] = providers.get(r.provider, 0) + 1
for p, n in sorted(providers.items()):
    print(f"  {p}: {n} scenes")

## Download best scene

In [ ]:
results.sort(key=lambda r: r.cloud_cover)
dl = client.download(
    results[:1],
    output_dir   = "./data/ch02/",
    bands        = ["B02","B03","B04","B08","B8A","B11"],
    post_process = ["cog"],
    skip_existing = True,
)
for d in dl:
    print(f"{'OK' if d.success else 'FAIL'}  {d.path}  {d.bytes_downloaded/1e6:.1f}MB")

## Time-series download

In [ ]:
quarters = [("2026-01-01","2026-03-31"),("2026-04-01","2026-06-30")]
for start, end in quarters:
    r = client.search(bbox=(-0.30,5.50,-0.05,5.70), date_range=(start,end),
                       cloud_cover_max=20)
    if r:
        print(f"{start[:7]}: {len(r)} scenes available, best cloud={r[0].cloud_cover:.1f}%")
    else:
        print(f"{start[:7]}: no scenes available")

## SAR search

In [ ]:
sar = client.search(bbox=(-0.30,5.50,-0.05,5.70),
                    date_range=("2026-06-01","2026-06-30"),
                    satellites=["Sentinel-1"], product_type="GRD")
print(f"SAR scenes: {len(sar)}")
for s in sar[:3]:
    print(f"  {s.datetime[:10]}  {s.provider}")